# PatchTST Fundamental — Evaluation & Backtest

**Purpose:** Validate fundamental model performance, run the quarterly top-N backtest,
and persist `eval-report.json`.

Sections:
1. Environment setup
2. Load saved model + metadata
3. Rebuild datasets (same config as training)
4. Bias & pipeline audit
5. Per-class evaluation metrics
6. Quarterly top-10 backtest (equal / confidence / rank)
7. Survivorship-bias caveat
8. Persist `eval-report.json`

In [ ]:
import json as _json
import random
import subprocess
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from torch.utils.data import DataLoader
from transformers import PatchTSTConfig

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IS_KAGGLE = Path('/kaggle/input').exists()

if IS_KAGGLE:
    REPO_URL    = 'https://github.com/AntonyAPT/SeniorProject.git'
    REPO_BRANCH = 'feature/zamnz_patchTST_weight_decay'
    REPO_DIR    = Path('/kaggle/working/SeniorProject')
    if not REPO_DIR.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    MODELS_DIR     = REPO_DIR / 'models'
    NOTEBOOK_DIR   = MODELS_DIR / 'notebooks' / 'fundamental'
    OHLCV_DATA_DIR = Path('/kaggle/input/datasets/kingz101/sp500-daily-raw')
    FUND_DATA_DIR  = Path('/kaggle/input/datasets/kingz101/sp500-quarterly-fundamental')
    SAVE_DIR       = Path('/kaggle/input/datasets/kingz101/patchtst-fundamental-saved-model')
else:
    NOTEBOOK_DIR = Path.cwd()
    if NOTEBOOK_DIR.name != 'fundamental':
        NOTEBOOK_DIR = Path('models/notebooks/fundamental').resolve()
    MODELS_DIR   = NOTEBOOK_DIR.parents[1]
    _local_data  = MODELS_DIR / 'data_raw'
    OHLCV_DATA_DIR = _local_data
    FUND_DATA_DIR  = _local_data
    SAVE_DIR     = NOTEBOOK_DIR / 'save_dir_fund'

FUND_CSV_FILENAME  = 'quarterly_fundamentals.csv'
OHLCV_CSV_FILENAME = 'historic_data_rows.csv'

sys.path.insert(0, str(MODELS_DIR))

from patchtst_lib.classification_head import PatchTSTClassifier
from patchtst_lib.fundamental.config import make_fundamental_patchtst_config
from patchtst_lib.fundamental.dataset import (
    QuarterlyFundamentalDataset,
    compute_fundamental_class_weights,
    split_fundamentals_by_date,
    summarize_fundamental_dataset,
)
from patchtst_lib.fundamental.features import (
    FUND_FEATURE_COLUMNS,
    build_feature_df,
    load_raw_fundamentals,
)
from patchtst_lib.labeling import LabelConfig
import patchtst_lib.fundamental.backtest as fbt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={device}')


In [ ]:
# ---- Load training metadata ----
meta_path = SAVE_DIR / 'fund_training_metadata.json'
if not meta_path.exists():
    raise FileNotFoundError(
        f'{meta_path} not found.  Run the training notebook first, then '
        f'pull artifacts with `bash pull_results.sh`.'
    )
with open(meta_path) as f:
    SAVED_META = _json.load(f)

print('Training metadata:')
for k, v in SAVED_META.items():
    if k not in ('class_weights', 'feature_columns', 'skipped_tickers'):
        print(f'  {k}: {v}')

# Reconstruct config from saved metadata.
CONTEXT_LENGTH    = SAVED_META['context_length']
PATCH_LENGTH      = SAVED_META['patch_length']
PATCH_STRIDE      = SAVED_META['patch_stride']
D_MODEL           = SAVED_META['d_model']
N_HEADS           = SAVED_META['num_attention_heads']
N_LAYERS          = SAVED_META['num_hidden_layers']
HORIZON           = SAVED_META['horizon']
N_CLASSES         = SAVED_META['n_classes']
FUND_FEATURES     = SAVED_META['feature_columns']
LABEL_RULE        = SAVED_META['label_rule']
LABEL_THRESHOLD   = SAVED_META['label_threshold']
PUBLISH_LAG_DAYS  = SAVED_META['publish_lag_days']
FORECAST_LAG_DAYS = SAVED_META['forecast_lag_days']
TRAIN_END         = SAVED_META['train_end']
VAL_END           = SAVED_META['val_end']

BATCH_SIZE = 256
NUM_WORKERS = 0


In [ ]:
class_weights = torch.tensor(SAVED_META['class_weights'], dtype=torch.float32)

patchtst_cfg = make_fundamental_patchtst_config(
    context_length=CONTEXT_LENGTH,
    patch_length=PATCH_LENGTH,
    patch_stride=PATCH_STRIDE,
    d_model=D_MODEL,
    num_attention_heads=N_HEADS,
    num_hidden_layers=N_LAYERS,
)

model = PatchTSTClassifier(
    config=patchtst_cfg,
    horizon=HORIZON,
    n_classes=N_CLASSES,
    class_weights=class_weights,
)

# Load weights from saved checkpoint.
ckpt_path = SAVE_DIR / 'patchtst_fund'
if ckpt_path.exists():
    from transformers import PatchTSTModel
    model.patchtst = PatchTSTModel.from_pretrained(str(ckpt_path))
    print(f'Loaded PatchTST encoder from {ckpt_path}')
else:
    print(f'WARNING: {ckpt_path} not found — using randomly initialised weights.')

model = model.to(device)
model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')


In [ ]:
_OHLCV_CANONICAL = {
    'date': 'Date', 'ticker': 'Ticker', 'open': 'Open',
    'high': 'High', 'low': 'Low', 'close': 'Close',
    'volume': 'Volume', 'sector': 'Sector',
}

def _normalize_ohlcv_cols(df):
    rename = {c: _OHLCV_CANONICAL[c.strip().lower()]
              for c in df.columns if c.strip().lower() in _OHLCV_CANONICAL}
    df = df.rename(columns=rename)
    df['Date'] = pd.to_datetime(df['Date'])
    return df

fund_csv = FUND_DATA_DIR / FUND_CSV_FILENAME
ohlcv_csv = OHLCV_DATA_DIR / OHLCV_CSV_FILENAME
if not fund_csv.exists():
    raise FileNotFoundError(f'{fund_csv} not found.')
if not ohlcv_csv.exists():
    raise FileNotFoundError(f'{ohlcv_csv} not found.')

raw_fund  = load_raw_fundamentals(str(fund_csv))
feat_df   = build_feature_df(raw_fund)
prices_df = _normalize_ohlcv_cols(pd.read_csv(ohlcv_csv))

label_config = LabelConfig(rule=LABEL_RULE, threshold=LABEL_THRESHOLD)
dataset_kwargs = dict(
    feature_columns=FUND_FEATURES,
    context_length=CONTEXT_LENGTH,
    label_config=label_config,
    publish_lag_days=PUBLISH_LAG_DAYS,
    forecast_lag_days=FORECAST_LAG_DAYS,
)

train_feat, val_feat, test_feat = split_fundamentals_by_date(
    feat_df, train_end=TRAIN_END, val_end=VAL_END, context_length=CONTEXT_LENGTH,
)

train_ds = QuarterlyFundamentalDataset(train_feat, prices_df, **dataset_kwargs)
val_ds   = QuarterlyFundamentalDataset(val_feat,   prices_df, **dataset_kwargs)
test_ds  = QuarterlyFundamentalDataset(test_feat,  prices_df, **dataset_kwargs)

for name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    print(summarize_fundamental_dataset(name, ds).to_string(index=False))


In [ ]:
def _run_inference(ds, batch_size=BATCH_SIZE):
    loader = DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS)
    logits_list, labels_list = [], []
    with torch.no_grad():
        for batch in loader:
            pv = batch['past_values'].to(device)
            out = model(past_values=pv)
            logits_list.append(out.logits.detach().cpu().float().numpy())
            labels_list.append(batch['labels'].numpy())
    logits_np = np.concatenate(logits_list, axis=0)  # (N, 1, 3)
    labels_np = np.concatenate(labels_list, axis=0)  # (N, 1)
    return logits_np, labels_np

print('Running inference on test set...')
TEST_LOGITS, TEST_LABELS = _run_inference(test_ds)
preds_flat  = np.argmax(TEST_LOGITS[:, 0, :], axis=-1)
labels_flat = TEST_LABELS[:, 0]

acc  = accuracy_score(labels_flat, preds_flat)
bacc = balanced_accuracy_score(labels_flat, preds_flat)
mf1  = f1_score(labels_flat, preds_flat, average='macro', zero_division=0)
print(f'Test  accuracy={acc:.4f}  balanced_acc={bacc:.4f}  macro_f1={mf1:.4f}')
print()
print(classification_report(labels_flat, preds_flat, target_names=['down','flat','up'], zero_division=0))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(labels_flat, preds_flat)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['down', 'flat', 'up']).plot(ax=ax, colorbar=False)
ax.set_title('Fundamental Model — Test Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
print('=' * 60)
print('AUDIT 1 — Publication-lag guard')
print('=' * 60)
print(f'''
Label formula (fundamental pipeline):

  decision_date = datadate + {PUBLISH_LAG_DAYS} days  (earliest 10-Q filing date)
  entry_close   = first close on or after decision_date
  exit_close    = first close on or after decision_date + {FORECAST_LAG_DAYS} days

  forward_return = (exit_close - entry_close) / entry_close
  label = UP if forward_return > {LABEL_THRESHOLD}
        = DOWN if forward_return < -{LABEL_THRESHOLD}
        = FLAT otherwise

No data from the forecast period leaks into feature computation.
''')

# Spot-check temporal ordering.
meta_0 = test_ds.metadata(0)
print(f'  Sample window: ticker={meta_0.ticker}')
print(f'    Quarter end     : {meta_0.context_end_quarter.date()}')
print(f'    Decision date   : {meta_0.decision_date.date()}  (+{PUBLISH_LAG_DAYS}d)')
print(f'    Forecast end    : {meta_0.forecast_end_date.date()}')
assert meta_0.decision_date > meta_0.context_end_quarter, 'decision_date <= quarter end!'
assert meta_0.forecast_end_date > meta_0.decision_date, 'forecast_end <= decision_date!'
print('  Temporal ordering check PASSED  [OK]')

print()
print('=' * 60)
print('AUDIT 2 — Survivorship bias caveat')
print('=' * 60)
print('''
  WARNING: The 90-ticker universe is today's S&P 500 constituents.
  Backtest numbers are inflated because delisted / bankrupt companies
  are not present.  Interpret all results with this in mind.
''')


In [ ]:
pred_df = fbt.build_prediction_df(
    logits=TEST_LOGITS,
    labels=TEST_LABELS,
    dataset=test_ds,
    prices_df=prices_df,
)
print(f'Prediction DataFrame: {len(pred_df):,} rows')
print(f'Decision-date range: {pred_df["decision_date"].min().date()} '
      f'-> {pred_df["decision_date"].max().date()}')

# --- Run three weighting strategies ---
WEIGHTING_SCHEMES = ['equal', 'confidence', 'rank']
TOP_N = 10
STARTING_CAPITAL = 1000.0

results_by_scheme = {}
summaries_by_scheme = {}

for w in WEIGHTING_SCHEMES:
    r = fbt.run_backtest(pred_df, starting_capital=STARTING_CAPITAL, top_n=TOP_N, weighting=w)
    results_by_scheme[w]   = r
    summaries_by_scheme[w] = fbt.summarize_results(r)

# Equity curve plot.
colors = {'equal': 'tab:blue', 'confidence': 'tab:orange', 'rank': 'tab:green'}
fig, ax = plt.subplots(figsize=(12, 5))
for w, r in results_by_scheme.items():
    eq = r['quarterly_equity']
    ret = r['total_return_pct']
    eq.plot(ax=ax, color=colors[w], linewidth=2, marker='o', markersize=4,
            label=f'{w}  ({ret:+.1f}%)')
ax.axhline(STARTING_CAPITAL, color='gray', linestyle='--', linewidth=0.8, label='baseline')
ax.set_title(f'Fundamental Quarterly Equity — Top-{TOP_N}')
ax.set_xlabel('Decision Date')
ax.set_ylabel('Portfolio Value ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import math

def _safe_float(v):
    if v is None: return None
    if isinstance(v, float) and math.isnan(v): return None
    return float(v)

best_scheme = max(results_by_scheme, key=lambda w: results_by_scheme[w]['total_return_pct'])

eval_report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'model_type': 'fundamental',
    'survivorship_bias_warning': (
        'Universe = today\'s S&P 500 constituents.  '
        'Delisted / bankrupt names are absent.'
    ),
    'dataset': {
        'tickers': feat_df['Ticker'].nunique(),
        'train_end': TRAIN_END,
        'val_end': VAL_END,
        'test_windows': len(test_ds),
        'feature_columns': FUND_FEATURES,
        'context_length': CONTEXT_LENGTH,
        'publish_lag_days': PUBLISH_LAG_DAYS,
        'forecast_lag_days': FORECAST_LAG_DAYS,
    },
    'test_metrics': {
        'accuracy': _safe_float(acc),
        'balanced_accuracy': _safe_float(bacc),
        'macro_f1': _safe_float(mf1),
        'label_rule': LABEL_RULE,
        'label_threshold': LABEL_THRESHOLD,
    },
    'backtest': {
        scheme: {
            'total_return_pct': _safe_float(s['total_return_pct']),
            'max_drawdown_pct': _safe_float(s['max_drawdown_pct']),
            'annualized_sharpe': _safe_float(s['annualized_sharpe']),
            'avg_quarterly_return_selected': _safe_float(s['avg_quarterly_return_selected']),
            'avg_quarterly_turnover': _safe_float(s['avg_quarterly_turnover']),
            'trading_quarters': s['trading_quarters'],
            'direction_accuracy_all': s['direction_accuracy_all'],
            'direction_accuracy_selected': s['direction_accuracy_selected'],
            'benchmark_return_pct': _safe_float(s['benchmark_return_pct']),
        }
        for scheme, s in summaries_by_scheme.items()
    },
    'best_backtest_scheme': best_scheme,
}

report_path = NOTEBOOK_DIR / 'eval-report.json'
with open(report_path, 'w') as f:
    _json.dump(eval_report, f, indent=2)
print(f'eval-report.json saved to {report_path}')
print(_json.dumps(eval_report, indent=2))
